# ส่วนที่ 4 · Fine-tune mT5 สรุปข่าวภาษาไทย (Abstractive)

**ผู้รับผิดชอบ:** คนที่ 4 (คู่ C) · **ส่ง:** โมเดลที่ดีที่สุด + ROUGE

**วิธีใช้:** Runtime → Change runtime type → **T4 GPU** แล้ว Runtime → **Run all**

- ข้อมูล: [ThaiSum](https://huggingface.co/datasets/pythainlp/thaisum) โหลดแบบ streaming ไม่ต้องโหลดทั้ง 350,000 ข่าว
- โมเดลเริ่มต้น: `csebuetnlp/mT5_multilingual_XLSum` เป็น mT5 ที่เคยเทรนสรุปข่าวหลายภาษารวมภาษาไทยมาแล้ว (XL-Sum, Hasan et al. 2021) เราเทรนต่อด้วยข่าวไทย
- ถ้าคนที่ 1 แจกไฟล์ train/val/test แล้ว ให้ตั้ง `TRAIN_CSV` / `VAL_CSV` / `TEST_CSV` (คอลัมน์ `body`, `summary`)

**คนที่ 5:** เปลี่ยน `MODEL_NAME = "facebook/mbart-large-50"` (ดูหมายเหตุ mBART ในช่องตั้งค่า) แล้ว Run all

In [ ]:
!pip -q install transformers datasets sentencepiece accelerate rouge_score pythainlp

## 1. ตั้งค่า (แก้แค่ช่องนี้)

In [ ]:
MODEL_NAME = "csebuetnlp/mT5_multilingual_XLSum"   # หรือ "google/mt5-small" (เริ่มจากศูนย์ ผลแย่กว่า แต่เร็ว)
# mBART (คนที่ 5): "facebook/mbart-large-50" + ตั้ง MBART_LANG = "th_TH"
MBART_LANG = None

# ข้อมูล: None = โหลด ThaiSum จาก Hugging Face · ใส่ path = ใช้ไฟล์ของคนที่ 1
TRAIN_CSV, VAL_CSV, TEST_CSV = None, None, None
TRAIN_SIZE = 5000    # จำนวนข่าวที่ใช้เทรน (Colab T4 ~5,000 ข่าว/epoch ประมาณ 30–40 นาที)
VAL_SIZE   = 300
TEST_SIZE  = 500     # ทุกโมเดลต้องใช้ test ชุดเดียวกัน

MAX_INPUT  = 512     # ความยาวข่าว (token) ที่ป้อนเข้า
MAX_TARGET = 128     # ความยาวบทสรุปสูงสุด
EPOCHS     = 2
LR         = 1e-4    # mt5-small เริ่มจากศูนย์ใช้ 5e-4 · mBART ใช้ 3e-5
BATCH_SIZE = 4
GRAD_ACC   = 4       # batch จริง = 4 x 4 = 16
NUM_BEAMS  = 4
OUTPUT_DIR = "mt5-thaisum"
SEED = 42

## 2. โหลดข้อมูล

In [ ]:
import pandas as pd
from datasets import load_dataset, Dataset

def from_thaisum(split, n, shuffle=False):
    ds = load_dataset("pythainlp/thaisum", split=split, streaming=True)
    if shuffle:
        ds = ds.shuffle(seed=SEED, buffer_size=10_000)   # สุ่มให้ได้ข่าวหลายสำนัก
    return pd.DataFrame(list(ds.take(n)))[["body", "summary"]]

def load(csv, split, n, shuffle=False):
    df = pd.read_csv(csv) if csv else from_thaisum(split, n, shuffle)
    df = df.dropna(subset=["body", "summary"])
    return df[(df.body.str.len() > 0) & (df.summary.str.len() > 0)].reset_index(drop=True)

train_df = load(TRAIN_CSV, "train", TRAIN_SIZE, shuffle=True)
val_df   = load(VAL_CSV, "validation", VAL_SIZE)
test_df  = load(TEST_CSV, "test", TEST_SIZE)
print("train", len(train_df), "| val", len(val_df), "| test", len(test_df))
print("\nตัวอย่างข่าว:", train_df.body[0][:200], "...\nบทสรุป:", train_df.summary[0])

## 3. Tokenize

In [ ]:
from transformers import AutoTokenizer
tok_kwargs = {"src_lang": MBART_LANG, "tgt_lang": MBART_LANG} if MBART_LANG else {}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, **tok_kwargs)

def prep(batch):
    x = tokenizer(batch["body"], max_length=MAX_INPUT, truncation=True)
    x["labels"] = tokenizer(text_target=batch["summary"], max_length=MAX_TARGET, truncation=True)["input_ids"]
    return x

train_ds = Dataset.from_pandas(train_df).map(prep, batched=True, remove_columns=["body", "summary"])
val_ds   = Dataset.from_pandas(val_df).map(prep, batched=True, remove_columns=["body", "summary"])

lens = [len(tokenizer(b).input_ids) for b in train_df.body[:200]]
print(f"ความยาวข่าวเฉลี่ย {sum(lens)/len(lens):.0f} token · ถูกตัดเกิน {MAX_INPUT}: {sum(l > MAX_INPUT for l in lens)/len(lens):.0%}")

## 4. ROUGE สำหรับภาษาไทย
ไลบรารี ROUGE แยกคำด้วยช่องว่าง ภาษาไทยจึงต้องตัดคำด้วย PyThaiNLP ก่อน (ดู README: ทุกคนใช้ฟังก์ชันเดียวกันนี้)

In [ ]:
from rouge_score import rouge_scorer
from pythainlp.tokenize import word_tokenize

class ThaiTokenizer:
    def tokenize(self, text):
        return [w for w in word_tokenize(text, engine="newmm") if w.strip()]

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=ThaiTokenizer())

def thai_rouge(preds, refs):
    s = [scorer.score(r, p) for p, r in zip(preds, refs)]
    return {k: round(100 * sum(x[k].fmeasure for x in s) / len(s), 2) for k in ["rouge1", "rouge2", "rougeL"]}

print(thai_rouge(["นายกรัฐมนตรีประชุมครม.วันนี้"], ["วันนี้นายกรัฐมนตรีเป็นประธานประชุมครม."]))

## 5. ลองก่อนเทรน (zero-shot)
เอาไว้เทียบว่าการ fine-tune ช่วยแค่ไหน

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "ไม่มี (ช้ามาก ให้เปิด T4 GPU)")

def summarize(texts, model, bs=8):
    out = []
    model.eval()
    gen_kwargs = {"forced_bos_token_id": tokenizer.lang_code_to_id[MBART_LANG]} if MBART_LANG else {}
    for i in range(0, len(texts), bs):
        enc = tokenizer(texts[i:i+bs], max_length=MAX_INPUT, truncation=True, padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            ids = model.generate(**enc, max_new_tokens=MAX_TARGET, num_beams=NUM_BEAMS,
                                 no_repeat_ngram_size=3, length_penalty=1.0, **gen_kwargs)
        out += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return out

from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
N0 = min(100, len(test_df))
zero = summarize(test_df.body[:N0].tolist(), model)
print("zero-shot ROUGE (100 ข่าวแรกของ test):", thai_rouge(zero, test_df.summary[:N0].tolist()))
print("\nตัวอย่าง:", zero[0])

## 6. Fine-tune
> mT5 ใช้ fp16 แล้วค่า loss มักเป็น NaN จึงเทรนแบบ fp32 + Adafactor + gradient checkpointing เพื่อให้พอกับ GPU ของ Colab
> ถ้าเจอ `CUDA out of memory` ให้ลด `BATCH_SIZE` เป็น 2 แล้วเพิ่ม `GRAD_ACC` เป็น 8

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, set_seed
set_seed(SEED)

args = Seq2SeqTrainingArguments(
    output_dir="checkpoints", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC, weight_decay=0.01,
    optim="adafactor", gradient_checkpointing=True,   # ประหยัดหน่วยความจำ GPU
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    fp16=False, logging_steps=50, report_to="none", seed=SEED)

trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                         data_collator=DataCollatorForSeq2Seq(tokenizer, model=model))
trainer.train()

## 7. วัดผลบน test (ส่งตารางนี้ให้คนที่ 7)

In [ ]:
preds = summarize(test_df.body.tolist(), trainer.model)
refs = test_df.summary.tolist()
scores = thai_rouge(preds, refs)
result = {"model": MODEL_NAME, **scores, "zero_shot_rouge_first100": thai_rouge(zero, refs[:N0]),
          "train_size": len(train_df), "test_size": len(test_df), "epochs": EPOCHS, "lr": LR,
          "max_input": MAX_INPUT, "num_beams": NUM_BEAMS}
result

### บทสรุปทั้งหมด (ส่งให้คนที่ 6 เช็กชื่อเฉพาะ และคนที่ 7 วิเคราะห์ข้อผิดพลาด)

In [ ]:
out = test_df.assign(pred=preds)
out["rougeL"] = [scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]
out.to_csv("predictions_mt5.csv", index=False)
pd.set_option("display.max_colwidth", 200)
print("บทสรุปที่ ROUGE-L ต่ำสุด 5 ข่าว:")
out.sort_values("rougeL")[["summary", "pred", "rougeL"]].head()

## 8. ลองสรุปข่าวใหม่

In [ ]:
news = """กรมอุตุนิยมวิทยาประกาศเตือนพายุฤดูร้อนบริเวณภาคเหนือและภาคตะวันออกเฉียงเหนือ ระหว่างวันที่ 10-12 นี้
จะมีพายุฝนฟ้าคะนอง ลมกระโชกแรง และลูกเห็บตกบางพื้นที่ ขอให้ประชาชนหลีกเลี่ยงการอยู่ในที่โล่งแจ้ง
และใต้ต้นไม้ใหญ่ ส่วนเกษตรกรควรเตรียมป้องกันความเสียหายของผลผลิต"""
summarize([news], trainer.model)[0]

## 9. บันทึกโมเดล
zip ไว้ดาวน์โหลด หรือคัดลอกเข้า Google Drive ของกลุ่ม · **ไม่ต้อง commit โมเดลเข้า GitHub** (ใหญ่ ~2 GB)

In [ ]:
import json, shutil
trainer.save_model(OUTPUT_DIR); tokenizer.save_pretrained(OUTPUT_DIR)
json.dump(result, open(f"{OUTPUT_DIR}/result.json", "w"), ensure_ascii=False, indent=2)
shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print("บันทึกแล้ว:", OUTPUT_DIR + ".zip")

# คัดลอกเข้า Drive (เอาคอมเมนต์ออกถ้าต้องการ)
# from google.colab import drive; drive.mount("/content/drive")
# shutil.copy(OUTPUT_DIR + ".zip", "/content/drive/MyDrive/")